# Shrinkage detector, round 2: two follow-up tests

Round 1 showed the detector **doesn't raise false alarms** when AIPW is set up properly (about 7% false alarms, near the 5% target).
But CausalPFN barely shrank, so we couldn't check whether the detector **catches** shrinkage.

This notebook runs two tests:

| Test | Question | Needs GPU? |
|---|---|---|
| **A. Fake fire (positive control)** | If a CFM shrinks by a known amount, does the detector catch it? | No |
| **B. Big effects (real CausalPFN)** | Does CausalPFN start shrinking when effects get very large? | Yes |

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
Test A takes about 15 to 20 minutes, Test B about 1 to 1.5 hours. Keep the tab open. Results save to Colab's local folder. If the runtime disconnects they are lost, so keep the tab open. At the end, download the CSV from the Files panel (folder icon on the left, then jackfruit_sim) if you want to keep it.

## 1. Setup

In [ ]:
!pip install -q causalpfn

import os, itertools, time, warnings
warnings.filterwarnings('ignore')
OUT_DIR = './jackfruit_sim'   # Colab's local folder (cleared if the runtime disconnects)
os.makedirs(OUT_DIR, exist_ok=True)
print('Saving results to', OUT_DIR)

## 2. Config
Set `RUN_A = False` if you already have Test A results and only want Test B.

In [ ]:
RUN_A = False   # Test A already done in round 2 (results were valid); set True to redo it
RUN_B = True
RUN_NAME = 'round3_krish'   # new name so the broken round-2 Test B results are not reused

# Test A: fake CFM that shrinks by a known fraction
A_EFFECTS = [1.0, 2.0, 4.0, 8.0]
A_NS      = [500, 2000]
A_SEEDS   = range(0, 30)
A_SHRINKS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]     # 0.2 = estimate pulled 20% toward zero

# Test B: real CausalPFN, much bigger effects than round 1 (which stopped at 8)
B_EFFECTS  = [1.0, 4.0, 8.0, 16.0, 32.0, 64.0, 128.0]
B_NS       = [500, 2000]
B_NOISE    = [1.0, 0.25]     # lower noise = effect is even bigger relative to noise
B_SEEDS    = range(0, 25)

A_CSV = f'{OUT_DIR}/{RUN_NAME}_A_positive_control.csv'
B_CSV = f'{OUT_DIR}/{RUN_NAME}_B_big_effects.csv'
print('Test A datasets:', len(A_EFFECTS)*len(A_NS)*len(A_SEEDS))
print('Test B datasets:', len(B_EFFECTS)*len(B_NS)*len(B_NOISE)*len(B_SEEDS))

## 3. Core code
Same simulator, AIPW, detector and answer key as round 1. Two changes:
- `simulate` takes a `noise_sd` knob.
- `fake_cfm` replaces the old mock: it shrinks the true effect by an exact, known fraction.

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

D = 6  # number of covariates

# ---------- 1. Data generator (we know the truth) ----------
def simulate(n, effect_size, overlap, seed, noise_sd=1.0):
    """effect_size = true ATE. overlap: 'good' or 'poor'. noise_sd = outcome noise."""
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n, D))
    # Nonlinear confounding: the same features drive treatment and outcome
    score = X[:, 0] - 0.5 * X[:, 1] + 0.8 * (X[:, 2] ** 2 - 1) + 0.5 * X[:, 0] * X[:, 3]
    strength = 0.6 if overlap == "good" else 2.0   # bigger = more extreme propensities
    e = expit(strength * score)
    T = rng.binomial(1, e)
    mu0 = 1.0 * X[:, 0] + np.sin(2 * X[:, 1]) + 0.7 * X[:, 2] ** 2 + 0.5 * X[:, 3] * X[:, 4]
    tau = effect_size * (1 + 0.5 * X[:, 4])          # heterogeneous effect, mean = effect_size
    Y = mu0 + T * tau + rng.normal(0, noise_sd, size=n)
    true_ate = float(tau.mean())                       # sample ATE (the "answer key")
    return X, T, Y, e, true_ate

# ---------- 2. AIPW with cross-fitting ----------
def _outcome_model(kind, seed):
    if kind == "flexible":
        return HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=seed)
    return LinearRegression()          # 'wrong': linear model on a nonlinear truth

def _prop_model(kind, seed):
    if kind == "flexible":
        return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=seed)
    return LogisticRegression(max_iter=1000)   # 'wrong': misses the squared / interaction terms

NUISANCE_SPECS = {
    "both_ok":       ("flexible", "flexible"),
    "outcome_wrong": ("wrong",    "flexible"),
    "prop_wrong":    ("flexible", "wrong"),
    "both_wrong":    ("wrong",    "wrong"),
}

def aipw(X, T, Y, spec, seed, folds=5, clip=0.01):
    """Returns ATE estimate, its standard error, and fitted propensities."""
    out_kind, prop_kind = NUISANCE_SPECS[spec]
    n = len(Y)
    mu0_hat, mu1_hat, e_hat = np.zeros(n), np.zeros(n), np.zeros(n)
    for tr, te in KFold(folds, shuffle=True, random_state=seed).split(X):
        pm = _prop_model(prop_kind, seed).fit(X[tr], T[tr])
        e_hat[te] = pm.predict_proba(X[te])[:, 1]
        for t, store in ((0, mu0_hat), (1, mu1_hat)):
            idx = tr[T[tr] == t]
            om = _outcome_model(out_kind, seed).fit(X[idx], Y[idx])
            store[te] = om.predict(X[te])
    e_c = np.clip(e_hat, clip, 1 - clip)
    psi = (mu1_hat - mu0_hat
           + T * (Y - mu1_hat) / e_c
           - (1 - T) * (Y - mu0_hat) / (1 - e_c))       # influence-function scores
    ate = psi.mean()
    se = psi.std(ddof=1) / np.sqrt(n)
    return float(ate), float(se), e_hat

# ---------- 3. The shrinkage detector ----------
def detector(cfm_ate, aipw_ate, aipw_se, e_hat, alpha_z=1.645, overlap_tol=0.05):
    """
    Flags shrinkage only if:
      (a) overlap is OK (else 'uncertain'),
      (b) CFM is closer to zero than AIPW (directional),
      (c) the gap is bigger than AIPW's own noise (one-sided z-test).
    """
    extreme = np.mean((e_hat < 0.02) | (e_hat > 0.98))
    if extreme > overlap_tol:
        return "uncertain", np.nan
    gap = aipw_ate - cfm_ate
    z = np.sign(aipw_ate) * gap / max(aipw_se, 1e-12)   # >0 means CFM sits closer to 0
    return ("flag" if z > alpha_z else "no_flag"), float(z)

# ---------- 4. Ground-truth label: did the CFM really shrink? ----------
def truly_shrunk(cfm_ate, true_ate, rel_tol=0.10):
    """CFM estimate is pulled toward zero by more than 10% of the true effect."""
    return bool(np.sign(true_ate) * (true_ate - cfm_ate) > rel_tol * abs(true_ate))

# ---------- Fake CFM for the positive control ----------
def fake_cfm(true_ate, shrink, rng, noise=0.05):
    """A fake CFM that shrinks the true effect by a known fraction (0.2 = 20% toward zero)."""
    return float(true_ate * (1 - shrink) + rng.normal(0, noise))


In [ ]:
import matplotlib.pyplot as plt

def scorecard(g):
    decided = g[g.decision != 'uncertain']
    pos, neg = decided[decided.truly_shrunk], decided[~decided.truly_shrunk]
    return pd.Series({
        'hit_rate':         (pos.decision == 'flag').mean() if len(pos) else np.nan,
        'false_alarm_rate': (neg.decision == 'flag').mean() if len(neg) else np.nan,
        'uncertain_rate':   (g.decision == 'uncertain').mean(),
        'n_shrunk':         len(pos),
        'n_not_shrunk':     len(neg),
    })

def done_keys(path, cols):
    if not os.path.exists(path):
        return set()
    prev = pd.read_csv(path)
    return set(map(tuple, prev[cols].drop_duplicates().values.tolist()))

## 4. Test A: fake fire (positive control)

A fake CFM shrinks the true effect by an exact fraction. AIPW (flexible models, good overlap) is fit once per dataset,
and then the detector is checked against every shrink level.

**What we want to see:** flag rate near 5% at shrink = 0, rising toward 100% as shrinkage grows, and rising faster with more data (n = 2000).

In [ ]:
from tqdm.auto import tqdm

if RUN_A:
    done = done_keys(A_CSV, ['effect_size', 'n', 'seed'])
    grid = [g for g in itertools.product(A_EFFECTS, A_NS, A_SEEDS) if g not in done]
    for es, n, seed in tqdm(grid, desc='Test A'):
        X, T, Y, e, ta = simulate(n, es, 'good', seed)
        a, se, eh = aipw(X, T, Y, 'both_ok', seed)
        rows = []
        for s in A_SHRINKS:
            c = fake_cfm(ta, s, np.random.default_rng(1000 + seed))
            d, z = detector(c, a, se, eh)
            rows.append(dict(effect_size=es, n=n, seed=seed, shrink=s, true_ate=ta, cfm_ate=c,
                             aipw_ate=a, aipw_se=se, z=z, decision=d, truly_shrunk=truly_shrunk(c, ta)))
        pd.DataFrame(rows).to_csv(A_CSV, mode='a', header=not os.path.exists(A_CSV), index=False)

HAVE_A = os.path.exists(A_CSV)
if HAVE_A:
    dfA = pd.read_csv(A_CSV)
    decA = dfA[dfA.decision != 'uncertain'].assign(flag=lambda d: d.decision == 'flag')
    print('Uncertain rate:', round((dfA.decision == 'uncertain').mean(), 3))
    print('Flag rate by shrink level (columns) and setting (rows):')
    display(decA.pivot_table(index=['n', 'effect_size'], columns='shrink', values='flag', aggfunc='mean').round(2))
else:
    print('Test A skipped (RUN_A = False).')

In [ ]:
if HAVE_A:
    fig, axes = plt.subplots(1, len(A_NS), figsize=(6 * len(A_NS), 4), sharey=True)
    for ax, n in zip(np.atleast_1d(axes), A_NS):
        for es in A_EFFECTS:
            m = decA[(decA.n == n) & (decA.effect_size == es)].groupby('shrink').flag.mean()
            ax.plot(m.index * 100, m.values, marker='o', label=f'true effect = {es:g}')
        ax.axhline(0.05, color='gray', ls='--', lw=1)
        ax.set_title(f'n = {n}'); ax.set_xlabel('how much the fake CFM shrinks (%)')
    np.atleast_1d(axes)[0].set_ylabel('detector flag rate'); np.atleast_1d(axes)[0].legend()
    plt.suptitle('Test A: does the detector catch known shrinkage?'); plt.tight_layout(); plt.show()
    print('Overall scorecard (answer key = shrunk by more than 10%):')
    display(scorecard(dfA).round(3).to_frame('Test A').T)

## 5. Test B: does real CausalPFN shrink big effects?

Round 1 only went up to effect = 8. Here we go up to 128, and also lower the noise so the effect is
even bigger compared to everything else in the data. AIPW uses the good setup (flexible models) only.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if RUN_B and device.type != 'cuda':
    print('WARNING: no GPU. Switch runtime to T4 GPU, otherwise Test B will be very slow.')

from causalpfn import ATEEstimator
from sklearn.base import clone

class SharedWeightsATE(ATEEstimator):
    """Fresh estimator per dataset, but the network weights are loaded onto the GPU only once.
    (fit() normally re-reads the checkpoint from disk every call. The round-2 bug came from
    estimate_ate() caching mu_0/mu_1, which fit() never resets, so a fresh object is still needed.)"""
    _cache = None
    def load_model(self):
        if SharedWeightsATE._cache is None:
            super().load_model()
            SharedWeightsATE._cache = (self.icl_model, self.max_feature_size, self.x_dim_transformer)
        self.icl_model, self.max_feature_size, xdt = SharedWeightsATE._cache
        self.x_dim_transformer = clone(xdt)

def cfm_ate(X, T, Y):
    # A FRESH estimator every call. Reusing one object returned the first dataset's answer forever (round 2 bug).
    est = SharedWeightsATE(device=device, verbose=False)
    est.fit(X.astype(np.float32), T.astype(np.float32), Y.astype(np.float32))
    return float(est.estimate_ate())

# Sanity check: two very different effects must give very different estimates
checks = []
for es in [4.0, 32.0]:
    X, T, Y, e, ta = simulate(1000, es, 'good', 123)
    t0 = time.time()
    c = cfm_ate(X, T, Y)
    checks.append(c)
    print(f'sanity check: true ATE = {ta:.2f}, CausalPFN ATE = {c:.2f}  ({time.time()-t0:.1f}s)')
assert abs(checks[1] - checks[0]) > 10, 'CausalPFN gave nearly the same answer for effect 4 and 32. Something is cached; stop and report.'
print('Sanity check passed: estimates change with the data.')

In [ ]:
if RUN_B:
    done = done_keys(B_CSV, ['effect_size', 'n', 'noise_sd', 'seed'])
    grid = [g for g in itertools.product(B_EFFECTS, B_NS, B_NOISE, B_SEEDS) if g not in done]
    for es, n, ns, seed in tqdm(grid, desc='Test B'):
        X, T, Y, e, ta = simulate(n, es, 'good', seed, noise_sd=ns)
        c = cfm_ate(X, T, Y)
        a, se, eh = aipw(X, T, Y, 'both_ok', seed)
        d, z = detector(c, a, se, eh)
        row = dict(effect_size=es, n=n, noise_sd=ns, seed=seed, true_ate=ta, cfm_ate=c,
                   aipw_ate=a, aipw_se=se, z=z, decision=d, truly_shrunk=truly_shrunk(c, ta),
                   y_sd=float(Y.std()))
        pd.DataFrame([row]).to_csv(B_CSV, mode='a', header=not os.path.exists(B_CSV), index=False)

dfB = pd.read_csv(B_CSV)
dfB['cfm_rel_bias']  = (dfB.cfm_ate - dfB.true_ate) / dfB.true_ate.abs()
dfB['aipw_rel_bias'] = (dfB.aipw_ate - dfB.true_ate) / dfB.true_ate.abs()
dfB['effect_over_ysd'] = dfB.true_ate.abs() / dfB.y_sd
print('Rows:', len(dfB))
print('Mean CausalPFN relative bias (negative = shrinks toward zero):')
display(dfB.pivot_table(index=['noise_sd', 'n'], columns='effect_size', values='cfm_rel_bias', aggfunc='mean').round(3))
print('Share of runs where CausalPFN shrank by more than 10%:')
display(dfB.pivot_table(index=['noise_sd', 'n'], columns='effect_size', values='truly_shrunk', aggfunc='mean').round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for (ns, n), g in dfB.groupby(['noise_sd', 'n']):
    m = g.groupby('effect_size')[['cfm_rel_bias', 'aipw_rel_bias']].mean()
    axes[0].plot(m.index, m.cfm_rel_bias * 100, marker='o', label=f'CausalPFN, noise={ns}, n={n}')
    axes[0].plot(m.index, m.aipw_rel_bias * 100, ls=':', color='gray')
axes[0].axhline(0, color='black', lw=1); axes[0].axhline(-10, color='red', ls='--', lw=1)
axes[0].set_xscale('log', base=2); axes[0].set_xlabel('true effect size')
axes[0].set_ylabel('relative bias (%)'); axes[0].set_title('Below the red line = shrinks more than 10%\n(gray dotted = AIPW)')
axes[0].legend(fontsize=8)

axes[1].scatter(dfB.effect_over_ysd, dfB.cfm_rel_bias * 100, s=8, alpha=0.4)
axes[1].axhline(0, color='black', lw=1); axes[1].axhline(-10, color='red', ls='--', lw=1)
axes[1].set_xscale('log'); axes[1].set_xlabel('effect size / SD of outcome')
axes[1].set_ylabel('CausalPFN relative bias (%)'); axes[1].set_title('Same thing, by how big the effect is vs the data')
plt.tight_layout(); plt.show()

### Detector on the real CausalPFN
If CausalPFN does shrink in some settings, this shows whether the detector caught it.

In [ ]:
display(dfB.groupby(['noise_sd', 'effect_size'])[['decision', 'truly_shrunk']].apply(scorecard).round(3))
dfB['aipw_covers'] = (dfB.aipw_ate - 1.96 * dfB.aipw_se <= dfB.true_ate) & (dfB.true_ate <= dfB.aipw_ate + 1.96 * dfB.aipw_se)
print('AIPW 95% coverage (should be near 0.95):', round(dfB.aipw_covers.mean(), 3))

## 6. Plain-English verdict

In [ ]:
if HAVE_A:
    print('=== Test A: does the detector catch shrinkage? ===')
    for s in [0.0, 0.1, 0.2, 0.3]:
        r = decA[decA.shrink == s].flag.mean()
        print(f'  CFM shrinks {int(s*100):>2}%  ->  detector flags {r*100:5.1f}% of the time')
    print()

print('Distinct CausalPFN answers across', len(dfB), 'runs:', dfB.cfm_ate.round(2).nunique(), '(should be in the hundreds)')
print('=== Test B: does real CausalPFN shrink big effects? ===')
by_es = dfB.groupby('effect_size').agg(bias=('cfm_rel_bias', 'mean'), shrunk=('truly_shrunk', 'mean'))
for es, r in by_es.iterrows():
    print(f'  effect {es:>6g}: average bias {r.bias*100:+6.1f}%, shrank >10% in {r.shrunk*100:5.1f}% of runs')
big = by_es[by_es.shrunk > 0.5]
if len(big):
    print(f'\nYES: CausalPFN shrinks in most runs once the effect reaches {big.index.min():g}.')
else:
    print('\nNO: CausalPFN did not shrink in most runs at any effect size tested.')
    print('That weakens the main hypothesis for this simulator. Worth raising with the guide.')